In [6]:
import einops
import torch

max_seq_len = 512
theta = 10_000
dim = 512

pairs_counts = torch.arange(0, dim, 2) / dim
t = torch.arange(max_seq_len)
inv_freqs = theta**-pairs_counts
freqs = einops.einsum(t, inv_freqs, "t, f -> t f")

cos, sin = freqs.cos(), freqs.sin()

In [4]:
import einops
import torch

max_seq_len = 512
theta = 10_000
dim = 512

d = torch.arange(0, dim, 2) / dim
freqs = torch.tensor(theta) ** -d
t = torch.arange(max_seq_len)

freqs = einops.einsum(t, freqs, "t, f -> t f")

d_cos, d_sin = torch.cos(freqs), torch.sin(freqs)

In [11]:
torch.stack((cos, sin)).shape

torch.Size([2, 512, 256])

In [12]:
x = torch.rand(10, 256, 512)

In [31]:
d_x1, d_x2 = einops.rearrange(x, "... (half_d xy) -> xy ... half_d", xy=2)

In [57]:
cos

tensor([[ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.5697,  0.5974,  ...,  1.0000,  1.0000,  1.0000],
        [-0.4161, -0.3509, -0.2863,  ...,  1.0000,  1.0000,  1.0000],
        ...,
        [ 0.9981,  0.6024, -0.7522,  ...,  0.9984,  0.9985,  0.9986],
        [ 0.4871, -0.3128, -0.9778,  ...,  0.9984,  0.9985,  0.9986],
        [-0.4717, -0.9588, -0.4159,  ...,  0.9984,  0.9985,  0.9986]])

In [40]:
x1, x2 = einops.rearrange(x, "... (half_d xy) -> xy ... half_d", xy=2).unbind(0)

In [46]:
x1 = torch.randint(0, 10, (1, 5, 10))
x2 = torch.randint(0, 10, (1, 5, 10))

In [50]:
x1

tensor([[[8, 9, 7, 8, 0, 2, 4, 4, 0, 2],
         [4, 4, 8, 0, 9, 8, 8, 9, 3, 2],
         [2, 1, 8, 6, 6, 8, 1, 7, 5, 8],
         [2, 6, 7, 6, 5, 9, 6, 0, 8, 8],
         [1, 4, 4, 8, 5, 6, 6, 4, 6, 0]]])

In [51]:
x2

tensor([[[1, 3, 9, 5, 1, 8, 5, 0, 8, 0],
         [9, 2, 4, 9, 3, 8, 5, 1, 1, 9],
         [8, 1, 2, 1, 2, 4, 5, 8, 0, 3],
         [1, 6, 2, 9, 7, 2, 5, 9, 9, 1],
         [4, 4, 7, 6, 8, 3, 5, 6, 3, 9]]])

In [59]:
x1, x2 = einops.rearrange(x, "... (half_d xy) -> xy ... half_d", xy=2).unbind(0)


In [61]:
seq_len = x.size(-2)
cos, sin = cos[:seq_len, :], sin[:seq_len, :]

In [62]:
x1_rot = cos * x1 - sin * x2
x2_rot = sin * x1 + cos * x2

In [64]:
torch.stack((x1_rot, x2_rot), dim=-1).flatten(-2).shape

torch.Size([10, 256, 512])

In [56]:
torch.stack((x1, x2), dim=-1).shape

torch.Size([1, 5, 10, 2])

In [49]:
torch.concat((x1, x2), dim=-1)

tensor([[[8, 9, 7, 8, 0, 2, 4, 4, 0, 2, 1, 3, 9, 5, 1, 8, 5, 0, 8, 0],
         [4, 4, 8, 0, 9, 8, 8, 9, 3, 2, 9, 2, 4, 9, 3, 8, 5, 1, 1, 9],
         [2, 1, 8, 6, 6, 8, 1, 7, 5, 8, 8, 1, 2, 1, 2, 4, 5, 8, 0, 3],
         [2, 6, 7, 6, 5, 9, 6, 0, 8, 8, 1, 6, 2, 9, 7, 2, 5, 9, 9, 1],
         [1, 4, 4, 8, 5, 6, 6, 4, 6, 0, 4, 4, 7, 6, 8, 3, 5, 6, 3, 9]]])